# OASIS-2 Minimal Colab Training

This notebook is a clean Colab-first path for OASIS-2 training.
It clones the repo, installs dependencies, runs the OASIS-2 training pipeline with live logs, and prints the saved summary.

In [10]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_colab_improved_v1'

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(name, path.exists(), path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT True /content/drive/MyDrive/Cerebrasensecloud
OASIS2_BUNDLE_ROOT True /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
RUNTIME_ROOT True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime


In [11]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = 'e577752'

def run_checked(cmd, *, cwd=None, label=None):
    print(f"RUNNING {label or cmd[0]}: {' '.join(cmd)}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    return completed

for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_checked(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_checked(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
run_checked(['python3', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

repo_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, label='git-rev-parse').stdout.strip()
print('repo_commit =', repo_commit)
print('required_commit =', REQUIRED_COMMIT)




RUNNING git-clone: git clone https://github.com/Billrichard209/cerebrasense.git /content/cerebrasense
Cloning into '/content/cerebrasense'...

RUNNING git-checkout-main: git checkout main
Your branch is up to date with 'origin/main'.

Already on 'main'

RUNNING pip-install: python3 -m pip install -r /content/cerebrasense/alz_backend/requirements-colab.txt

RUNNING git-rev-parse: git rev-parse HEAD
b3a7bc00fc6927a253a4606461bbc23237c231f1

repo_commit = b3a7bc00fc6927a253a4606461bbc23237c231f1
required_commit = e577752


In [13]:
from pathlib import Path
import pandas as pd

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
print('run_root =', run_root.exists(), run_root)

for p in [
    run_root / 'metrics' / 'epoch_metrics.csv',
    run_root / 'checkpoints' / 'best_model.pt',
    run_root / 'checkpoints' / 'last_model.pt',
    run_root / 'reports' / 'colab_run_summary.json',
]:
    print(p, '->', p.exists())

metrics = run_root / 'metrics' / 'epoch_metrics.csv'
if metrics.exists():
    df = pd.read_csv(metrics)
    print(df.tail())



run_root = True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1
/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1/metrics/epoch_metrics.csv -> True
/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1/checkpoints/best_model.pt -> True
/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1/checkpoints/last_model.pt -> True
/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1/reports/colab_run_summary.json -> True
   epoch  learning_rate  train_loss  val_loss  accuracy     auroc  precision  \
0     27   1.296818e-05    0.157111  0.248939  0.611111  0.577503   0.571429   
1     28   7.341523e-06    0.160690  0.452782  0.518519  0.565158   0.509434   
2     29   3.277860e-06    0.151637  0.281233  0.648148  0.618656   0.595238   
3     30   8.

In [12]:
import subprocess
import sys
from pathlib import Path
import torch

print('cuda_available =', torch.cuda.is_available())
print('gpu_name =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached to this Colab runtime. Stop here and switch Runtime -> Change runtime type -> GPU.')

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
last_checkpoint = run_root / 'checkpoints' / 'last_model.pt'
resume_args = []
if last_checkpoint.exists():
    print('Resuming from existing checkpoint:', last_checkpoint, flush=True)
    resume_args = ['--resume-from', str(last_checkpoint)]

cmd = [
    sys.executable,
    'scripts/train_oasis2_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--bundle-root', str(OASIS2_BUNDLE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', '30',
    '--batch-size', '2',
    '--gradient-accumulation-steps', '4',
    '--num-workers', '2',
    '--image-size', '96', '96', '96',
    '--learning-rate', '3e-4',
    '--weight-decay', '0.01',
    '--scheduler', 'cosine',
    '--step-size', '30',
    '--weighted-sampling',
    '--seed', '42',
    '--split-seed', '42',
    '--device', 'auto',
] + resume_args

print('RUNNING:', ' '.join(cmd), flush=True)
process = subprocess.Popen(
    cmd,
    cwd=BACKEND_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'OASIS-2 training runner failed with exit code {return_code}')

print('OASIS-2 training runner completed successfully.', flush=True)



cuda_available = True
gpu_name = Tesla T4
Resuming from existing checkpoint: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1/checkpoints/last_model.pt
RUNNING: /usr/bin/python3 scripts/train_oasis2_colab.py --project-root /content/cerebrasense/alz_backend --runtime-root /content/drive/MyDrive/Cerebrasensecloud/backend_runtime --bundle-root /content/drive/MyDrive/Cerebrasensecloud/OASIS-2 --run-name oasis2_colab_improved_v1 --epochs 30 --batch-size 2 --gradient-accumulation-steps 4 --num-workers 2 --image-size 96 96 96 --learning-rate 3e-4 --weight-decay 0.01 --scheduler cosine --step-size 30 --weighted-sampling --seed 42 --split-seed 42 --device auto --resume-from /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1/checkpoints/last_model.pt
Starting OASIS-2 Colab bundle pipeline
Resolved OASIS-2 bundle root: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
Runtime data root: /content

In [9]:
from pathlib import Path
import json
import pandas as pd

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
summary_path = run_root / 'reports' / 'colab_run_summary.json'
metrics_path = run_root / 'metrics' / 'epoch_metrics.csv'

print('run_root =', run_root.exists(), run_root)
print('summary_path =', summary_path, '->', summary_path.exists())
print('metrics_path =', metrics_path, '->', metrics_path.exists())

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2))
    print('training_ready =', summary.get('training_ready'))
    print('training_started =', summary.get('training_started'))
    print('blocked_reason =', summary.get('blocked_reason'))
    print('run_root =', summary.get('run_root'))
    print('best_checkpoint =', summary.get('best_checkpoint'))
else:
    print('No completed colab_run_summary.json yet. Showing current epoch metrics instead.')

if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    print(metrics.tail())



run_root = True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1
summary_path = /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1/reports/colab_run_summary.json -> True
metrics_path = /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_improved_v1/metrics/epoch_metrics.csv -> True
{
  "generated_at": "2026-05-07T12:15:06.570378+00:00",
  "bundle_root": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2",
  "bundle_source_for_checks": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2",
  "source_root_for_training": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2",
  "runtime_data_root": "/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data",
  "runtime_outputs_root": "/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs",
  "metadata_template_source_path": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2/backend_reference/o

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')



True
Tesla T4


In [ ]:
import os
os.chdir('/content/cerebrasense/alz_backend')

!/usr/bin/python3 scripts/train_oasis2_colab.py \
    --project-root /content/cerebrasense/alz_backend \
    --runtime-root /content/drive/MyDrive/Cerebrasensecloud/backend_runtime \
    --bundle-root /content/drive/MyDrive/Cerebrasensecloud/OASIS-2 \
    --run-name oasis2_monotonic_v2 \
    --epochs 50 \
    --batch-size 2 \
    --gradient-accumulation-steps 4 \
    --num-workers 2 \
    --image-size 96 96 96 \
    --learning-rate 2e-4 \
    --weight-decay 0.01 \
    --scheduler cosine \
    --weighted-sampling \
    --seed 42 \
    --split-seed 42 \
    --device auto \
    --config configs/oasis2_train.yaml



Starting OASIS-2 Colab bundle pipeline
Resolved OASIS-2 bundle root: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
Runtime data root: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data
Runtime outputs root: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs
Inspecting uploaded OASIS-2 bundle
Upload bundle inspection status: pass
Resolving OASIS-2 metadata template source
Resolved metadata template source: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2/backend_reference/oasis2_metadata_template.csv
Running preflight runtime refresh from Drive bundle
Refreshing runtime artifacts from source root: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
Copying OASIS-2 metadata template into runtime data root
Runtime metadata template path: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data/interim/oasis2_metadata_template.csv
Runtime metadata template has no labels; resolving official OASIS-2 demographics source
Merging official OASIS-2 demographics from: